# Índice geoespacial de criticidad urbana — Guayaquil
## Notebook piloto (prueba de concepto)

**Proyecto:** Índice geoespacial de criticidad urbana basado en peligrosidad, exposición y vulnerabilidad
**Institución:** Universidad de Guayaquil · Facultad de Ingeniería Industrial · Carrera de Sistemas de la Información
**Metodología:** CRISP-DM (las secciones del notebook siguen sus fases)

---

### ¿Qué hace este notebook?

Implementa, de punta a punta y con **datos sintéticos**, el flujo descrito en el anteproyecto:

| Paso | Contenido | Requisito del anteproyecto |
|---|---|---|
| 1–2 | Configuración y parámetros | — |
| 3 | Carga de datos (sintéticos, CSV o Supabase) | RF-01, RF-02, RF-03 |
| 4 | Cuadrícula de 500 × 500 m y *join* espacial | RF-04 |
| 5 | Agregación por unidad de análisis | — |
| 6 | Normalización (min-max y z-score) | RF-05 |
| 7 | Cálculo del índice de criticidad | RF-06 |
| 8 | Ranking y categorías por percentil | RF-07 |
| 9 | Análisis de sensibilidad (±10 %, ±20 %, ±30 %) con Spearman | RF-08, H3 |
| 10 | Autocorrelación espacial: Moran global, Gi\* y LISA | Sección 8.5 |
| 11 | Validación frente a puntos críticos y superposición vial–criminal | H1, H2 |
| 12 | Mapas estáticos e interactivos (Folium) | RF-09 |
| 13 | Exportación (archivos y Supabase) y registro de ejecución | RF-10, RF-11 |
| 14 | *Extensión opcional:* modelo predictivo base + informe con LLM | Borrador de datos |

> ⚠️ **Importante.** Todos los datos de este piloto son **ficticios**: se generan al azar para probar que el código funciona. Los mapas, rankings y correlaciones **no describen la realidad de Guayaquil** y no deben citarse como resultados. En la sección 3 se indica cómo reemplazarlos por los datos oficiales (ATM, Policía Nacional, INEC).

## 1. Comprensión del negocio (CRISP-DM · fase 1)

**Pregunta principal.** ¿Cómo combinar variables espaciales de siniestralidad vial y criminalidad para construir un índice geoespacial de criticidad urbana, interpretable y replicable?

**Marco conceptual.** La criticidad de cada zona se modela como una combinación de cuatro dimensiones:

- **Peligrosidad vial:** siniestros con lesionados y con fallecidos.
- **Peligrosidad criminal:** homicidios intencionales y robos.
- **Exposición:** densidad poblacional, puntos de interés (POIs) y flujo vehicular.
- **Vulnerabilidad:** pobreza, vivienda precaria, carencia de servicios básicos y población vulnerable (adultos mayores y menores).

**Unidad de análisis.** Celdas de 500 × 500 m (alternativa a los 67 circuitos de planificación policial, que se pueden cargar cuando se disponga de su geometría).

**Hipótesis que este notebook permite contrastar (con datos reales):**

- **H1:** las zonas de alta criticidad coinciden con puntos críticos conocidos.
- **H2:** existe superposición espacial entre siniestralidad vial y criminalidad.
- **H3:** el ranking es estable (Spearman alto) ante variaciones de ±20 % en los pesos.

## 2. Preparación del entorno y parámetros

Las librerías `geopandas`, `folium`, `scikit-learn` y `matplotlib` ya vienen en Colab. Solo se instalan `esda` y `libpysal` (estadística espacial).

In [ ]:
!pip -q install esda libpysal

In [ ]:
import warnings, logging, json
from pathlib import Path

import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import folium
from folium.plugins import HeatMap
from shapely.geometry import box
from scipy.stats import spearmanr

from libpysal.weights import Queen
from esda.moran import Moran, Moran_Local, Moran_BV
from esda.getisord import G_Local

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 140)
plt.rcParams["figure.dpi"] = 110

### 2.1 Parámetros del piloto

Todo lo que se pueda ajustar está concentrado en esta celda. **Los pesos son provisionales**: en el proyecto deben definirse y justificarse con el análisis multicriterio (objetivo específico 2).

In [ ]:
# --- Reproducibilidad -------------------------------------------------------
SEED = 42
np.random.seed(SEED)
rng = np.random.default_rng(SEED)

# --- Sistemas de coordenadas ------------------------------------------------
CRS_GEO = "EPSG:4326"   # latitud / longitud (WGS84)
CRS_UTM = "EPSG:32717"  # UTM zona 17 Sur (metros) -> adecuado para Guayaquil

# --- Unidad de análisis -----------------------------------------------------
CELL_SIZE = 500                          # metros
BBOX = (-79.98, -2.30, -79.86, -2.08)    # (lon_min, lat_min, lon_max, lat_max), área aproximada del piloto

# --- Origen de datos --------------------------------------------------------
USAR_DATOS_SINTETICOS = True   # False -> se usan datos reales (ver secciones 3.3 y 3.4)
FUENTE_REAL = "supabase"       # si USAR_DATOS_SINTETICOS=False: "supabase" o "csv"

# --- Tablas en Supabase (PostgreSQL) -----------------------------------------
TABLA_SINIESTROS = "nombreDetabla"        # <- reemplazar por el nombre real de tu tabla de accidentes
TABLA_DELITOS    = "delitos"
TABLA_CONTEXTO   = "contexto_celdas"
TABLA_PUNTOS     = "puntos_criticos"
TABLA_RANKING    = "ranking_criticidad"   # destino de los resultados (sección 13)
GUARDAR_EN_SUPABASE = False               # True -> sube el ranking al final del notebook
# Si tus columnas se llaman distinto, mapea {nombre_en_tu_tabla: nombre_esperado}
MAPA_COLUMNAS_SINIESTROS = {}             # ej.: {"latitud": "lat", "longitud": "lon", "heridos": "lesionados"}

# --- Pesos de las dimensiones (PROVISIONALES) -------------------------------
PESOS_DIM = {
    "peligrosidad_vial":     0.30,
    "peligrosidad_criminal": 0.30,
    "exposicion":            0.20,
    "vulnerabilidad":        0.20,
}

# --- Variables por dimensión (matriz de operacionalización, Tabla 2) --------
DIMENSIONES = {
    "peligrosidad_vial":     ["siniestros_lesionados", "siniestros_fallecidos"],
    "peligrosidad_criminal": ["homicidios", "robos"],
    "exposicion":            ["densidad_hab_km2", "pois", "flujo_veh_dia"],
    "vulnerabilidad":        ["pobreza_pct", "vivienda_precaria_pct", "sin_servicios_pct", "pob_vulnerable_pct"],
}
VARS_CONTEO = ["siniestros_lesionados", "siniestros_fallecidos", "homicidios", "robos", "pois"]
LOG_CONTEOS = True   # log(1+x) a los conteos antes de normalizar (reduce el efecto de valores extremos)

# --- Criterios de evaluación -----------------------------------------------
DELTAS = (0.10, 0.20, 0.30)    # variaciones de pesos para la sensibilidad
UMBRAL_SPEARMAN = 0.90         # supuesto del piloto para considerar "alta estabilidad"
PERMUTACIONES = 999            # permutaciones para pruebas de significancia espacial

# --- Salidas ----------------------------------------------------------------
OUT = Path("salidas_piloto"); OUT.mkdir(exist_ok=True)

logging.basicConfig(
    level=logging.INFO, force=True,
    format="%(asctime)s | %(levelname)s | %(message)s",
    handlers=[logging.FileHandler(OUT / "ejecucion.log", mode="w", encoding="utf-8"), logging.StreamHandler()],
)
log = logging.getLogger("criticidad")
log.info("Parámetros: celda=%sm | pesos=%s | datos_sinteticos=%s", CELL_SIZE, PESOS_DIM, USAR_DATOS_SINTETICOS)

## 3. Comprensión y carga de datos (CRISP-DM · fases 2 y 3)

### 3.1 Estructura esperada de los datos

Para pasar de datos sintéticos a reales, cada fuente debe respetar estas columnas mínimas.

| Archivo | Fuente | Columnas mínimas |
|---|---|---|
| `siniestros.csv` | ATM / Observatorio de Movilidad | `fecha`, `lat`, `lon`, `lesionados`, `fallecidos` |
| `delitos.csv` | Policía Nacional / ECU911 | `fecha`, `lat`, `lon`, `tipo` (`homicidio`, `robo_persona`, `robo_domicilio`) |
| `contexto_celdas.csv` | INEC, GAD, OSM, ATM | `id_celda`, `densidad_hab_km2`, `pois`, `flujo_veh_dia`, `pobreza_pct`, `vivienda_precaria_pct`, `sin_servicios_pct`, `pob_vulnerable_pct` |
| `puntos_criticos.csv` | ATM / Policía | `nombre`, `lat`, `lon` |

Los datos del INEC suelen venir por sector o zona censal, no por celda; en el proyecto real habrá que **reasignarlos a la cuadrícula** (por ejemplo, ponderando por área de intersección).

### 3.2 Generador de datos sintéticos

Se simulan puntos de siniestros y delitos concentrados alrededor de centros ficticios, con **solape parcial** entre ambos para poder probar el análisis de superposición (H2).

In [ ]:
def generar_datos_sinteticos(n_vial=6000, n_crim=9000):
    # Devuelve siniestros, delitos y los centros ficticios de concentración.
    lon_min, lat_min, lon_max, lat_max = BBOX

    def centros(k):
        return np.column_stack([
            rng.uniform(lon_min + 0.015, lon_max - 0.015, k),
            rng.uniform(lat_min + 0.03,  lat_max - 0.03,  k),
        ])

    c_vial, c_crim = centros(5), centros(5)
    c_crim[:2] = c_vial[:2] + rng.normal(0, 0.004, (2, 2))   # 2 centros compartidos -> solape vial-criminal

    def puntos(n, cen, frac, sigma):
        n_c = int(n * frac)
        xy_c = cen[rng.integers(0, len(cen), n_c)] + rng.normal(0, sigma, (n_c, 2))
        xy_u = np.column_stack([rng.uniform(lon_min, lon_max, n - n_c), rng.uniform(lat_min, lat_max, n - n_c)])
        xy = np.vstack([xy_c, xy_u])
        xy[:, 0] = np.clip(xy[:, 0], lon_min, lon_max - 1e-6)
        xy[:, 1] = np.clip(xy[:, 1], lat_min, lat_max - 1e-6)
        return xy

    def fechas(n):  # 2023-01-01 a 2026-06-30
        return pd.Timestamp("2023-01-01") + pd.to_timedelta(rng.integers(0, 1277, n), unit="D")

    xy = puntos(n_vial, c_vial, 0.70, 0.006)
    vial = pd.DataFrame({"fecha": fechas(n_vial), "lat": xy[:, 1], "lon": xy[:, 0],
                         "lesionados": rng.poisson(1.2, n_vial), "fallecidos": rng.binomial(1, 0.07, n_vial)})

    xy = puntos(n_crim, c_crim, 0.75, 0.006)
    crim = pd.DataFrame({"fecha": fechas(n_crim), "lat": xy[:, 1], "lon": xy[:, 0],
                         "tipo": rng.choice(["homicidio", "robo_persona", "robo_domicilio"], n_crim, p=[0.12, 0.58, 0.30])})
    return vial, crim, c_vial, c_crim


def generar_contexto_sintetico(grid, c_crim):
    # Contexto por celda: la vulnerabilidad y la exposición aumentan cerca de los centros ficticios.
    cen = grid.geometry.centroid.to_crs(CRS_GEO)
    xy = np.column_stack([cen.x, cen.y])
    d = np.min(np.linalg.norm(xy[:, None, :] - c_crim[None, :, :], axis=2), axis=1)
    cerca = np.exp(-d / 0.03)
    n = len(grid)
    pct = lambda v: np.clip(v, 0, 100)
    return pd.DataFrame({
        "id_celda": grid["id_celda"].values,
        "densidad_hab_km2": np.clip(rng.normal(6000, 2500, n) + 9000 * cerca, 200, None),
        "pois": rng.poisson(3 + 25 * cerca),
        "flujo_veh_dia": np.clip(rng.normal(8000, 3000, n) + 12000 * cerca, 0, None),
        "pobreza_pct": pct(rng.normal(25, 8, n) + 30 * cerca),
        "vivienda_precaria_pct": pct(rng.normal(12, 6, n) + 25 * cerca),
        "sin_servicios_pct": pct(rng.normal(15, 7, n) + 25 * cerca),
        "pob_vulnerable_pct": pct(rng.normal(22, 4, n) + 6 * cerca),
    })

### 3.3 Carga de datos reales (opcional)

Si los datos están en archivos CSV, pon `USAR_DATOS_SINTETICOS = False` y `FUENTE_REAL = "csv"`, monta Google Drive y ajusta `RUTA_DATOS`. Si están en Supabase, ve a la sección 3.4.

In [ ]:
RUTA_DATOS = Path("/content/drive/MyDrive/criticidad_gye")   # carpeta con los CSV reales

def cargar_datos_csv():
    vial = pd.read_csv(RUTA_DATOS / "siniestros.csv", parse_dates=["fecha"])
    crim = pd.read_csv(RUTA_DATOS / "delitos.csv", parse_dates=["fecha"])
    contexto = pd.read_csv(RUTA_DATOS / "contexto_celdas.csv")
    puntos = pd.read_csv(RUTA_DATOS / "puntos_criticos.csv")
    # Validaciones básicas de calidad (el LLM y los modelos dependen de esto)
    for nombre, df, cols in [("siniestros", vial, ["fecha", "lat", "lon", "lesionados", "fallecidos"]),
                             ("delitos", crim, ["fecha", "lat", "lon", "tipo"])]:
        faltan = set(cols) - set(df.columns)
        assert not faltan, f"{nombre}: faltan columnas {faltan}"
        df.dropna(subset=["lat", "lon", "fecha"], inplace=True)
    return vial, crim, contexto, puntos

# Para usar datos de Drive, descomentar:
# from google.colab import drive; drive.mount("/content/drive")

### 3.4 Carga y guardado de datos en Supabase (PostgreSQL)

Supabase expone cada tabla de PostgreSQL mediante una API. Aquí se usa el cliente oficial (`supabase-py`) para **cargar** las tablas del proyecto y **guardar** los resultados.

**Antes de ejecutar:**

1. En Colab, panel izquierdo → 🔑 *Secretos*, crea `SUPABASE_URL` y `SUPABASE_KEY` y activa el acceso para este notebook. Así las credenciales **no quedan escritas en el código** (si compartes o subes el notebook a GitHub, no se filtran).
2. Usa la clave `anon` para **leer** (con una política de lectura en la tabla). Para **escribir** hace falta una política de inserción o la clave `service_role`; esta última da acceso total, así que trátala como contraseña y no la compartas.
3. Supabase devuelve por defecto **máximo 1000 filas por consulta**. La función de carga pagina automáticamente; sin eso, tus tablas de miles de registros quedarían truncadas sin ningún error.

In [ ]:
!pip -q install supabase

In [ ]:
import os
from supabase import create_client, Client

def _cliente_supabase() -> Client:
    # Lee las credenciales desde los Secretos de Colab (o variables de entorno).
    try:
        from google.colab import userdata
        url, key = userdata.get("SUPABASE_URL"), userdata.get("SUPABASE_KEY")
    except Exception:
        url, key = os.environ.get("SUPABASE_URL"), os.environ.get("SUPABASE_KEY")
    if not url or not key:
        raise RuntimeError("Faltan SUPABASE_URL / SUPABASE_KEY en los Secretos de Colab.")
    return create_client(url, key)


def cargar_tabla_supabase(tabla, tam_pagina=1000):
    # Equivale a SELECT * FROM tabla, trayendo todas las filas por páginas.
    cli, filas, desde = _cliente_supabase(), [], 0
    print(f"📡 Consultando Supabase: {tabla} ...")
    while True:
        r = cli.table(tabla).select("*").range(desde, desde + tam_pagina - 1).execute()
        filas.extend(r.data)
        if len(r.data) < tam_pagina:
            break
        desde += tam_pagina
    df = pd.DataFrame(filas)
    log.info("Supabase: %s -> %d filas", tabla, len(df))
    return df


def guardar_tabla_supabase(df, tabla, lote=500, columna_unica=None):
    # Inserta un DataFrame por lotes. Con columna_unica hace upsert (actualiza si ya existe).
    cli, d = _cliente_supabase(), df.copy()
    for c in d.select_dtypes(include=["datetime", "datetimetz"]).columns:
        d[c] = d[c].dt.strftime("%Y-%m-%dT%H:%M:%S")
    d = d.astype(object).where(d.notna(), None)            # NaN -> NULL; tipos compatibles con JSON
    registros = d.to_dict("records")
    for i in range(0, len(registros), lote):
        t = cli.table(tabla)
        if columna_unica:
            t.upsert(registros[i:i + lote], on_conflict=columna_unica).execute()
        else:
            t.insert(registros[i:i + lote]).execute()
    log.info("Supabase: %d filas guardadas en %s", len(registros), tabla)
    print(f"✅ {len(registros)} filas guardadas en '{tabla}'")


def cargar_datos_supabase():
    vial = cargar_tabla_supabase(TABLA_SINIESTROS).rename(columns=MAPA_COLUMNAS_SINIESTROS)
    crim = cargar_tabla_supabase(TABLA_DELITOS)
    contexto = cargar_tabla_supabase(TABLA_CONTEXTO)
    puntos = cargar_tabla_supabase(TABLA_PUNTOS)
    for nombre, df, cols in [("siniestros", vial, ["fecha", "lat", "lon", "lesionados", "fallecidos"]),
                             ("delitos", crim, ["fecha", "lat", "lon", "tipo"])]:
        faltan = set(cols) - set(df.columns)
        assert not faltan, f"{nombre}: faltan columnas {faltan}. Revisa MAPA_COLUMNAS_SINIESTROS o los nombres en Supabase."
        df["fecha"] = pd.to_datetime(df["fecha"], errors="coerce")
        df.dropna(subset=["lat", "lon", "fecha"], inplace=True)
    return vial, crim, contexto, puntos


def cargar_datos_reales():
    return cargar_datos_supabase() if FUENTE_REAL == "supabase" else cargar_datos_csv()

**Prueba rápida de conexión** (solo la tabla de accidentes, para ver sus columnas antes de correr todo el flujo). Descomenta las líneas cuando tengas los secretos configurados:

In [ ]:
# df_accidentes = cargar_tabla_supabase(TABLA_SINIESTROS)
# display(df_accidentes.head()); print(df_accidentes.dtypes)

# Para SUBIR datos sintéticos de prueba a una tabla (la tabla debe existir con columnas compatibles):
# guardar_tabla_supabase(siniestros, "siniestros_prueba")

## 4. Cuadrícula de análisis y *join* espacial (RF-04)

Se trabaja en **UTM 17S (metros)** para que las celdas midan exactamente 500 × 500 m, y luego se vuelve a lat/lon solo para los mapas.

In [ ]:
def crear_cuadricula(bbox, size_m):
    minx, miny, maxx, maxy = gpd.GeoSeries([box(*bbox)], crs=CRS_GEO).to_crs(CRS_UTM).total_bounds
    xs, ys = np.arange(minx, maxx, size_m), np.arange(miny, maxy, size_m)
    celdas = [box(x, y, x + size_m, y + size_m) for x in xs for y in ys]
    return gpd.GeoDataFrame({"id_celda": np.arange(len(celdas))}, geometry=celdas, crs=CRS_UTM)

def a_geodf(df):
    return gpd.GeoDataFrame(df.copy(), geometry=gpd.points_from_xy(df["lon"], df["lat"]), crs=CRS_GEO).to_crs(CRS_UTM)

grid = crear_cuadricula(BBOX, CELL_SIZE)
log.info("Cuadrícula creada: %d celdas de %dx%d m", len(grid), CELL_SIZE, CELL_SIZE)

if USAR_DATOS_SINTETICOS:
    siniestros, delitos, c_vial, c_crim = generar_datos_sinteticos()
    contexto = generar_contexto_sintetico(grid, c_crim)
    # "Puntos críticos conocidos" ficticios: los centros de concentración
    puntos_criticos = pd.DataFrame({
        "nombre": [f"Punto vial {i+1}" for i in range(len(c_vial))] + [f"Punto criminal {i+1}" for i in range(len(c_crim))],
        "lon": np.r_[c_vial[:, 0], c_crim[:, 0]], "lat": np.r_[c_vial[:, 1], c_crim[:, 1]],
    })
else:
    siniestros, delitos, contexto, puntos_criticos = cargar_datos_reales()

print(f"Siniestros: {len(siniestros):,} | Delitos: {len(delitos):,} | Celdas: {len(grid):,}")
siniestros.head()

In [ ]:
# Join espacial: cada incidente se asigna a la celda que lo contiene
def asignar_celda(df):
    g = gpd.sjoin(a_geodf(df), grid[["id_celda", "geometry"]], how="inner", predicate="within")
    return pd.DataFrame(g.drop(columns=["geometry", "index_right"]))

sin_cel = asignar_celda(siniestros)
del_cel = asignar_celda(delitos)

perdidos_v, perdidos_d = len(siniestros) - len(sin_cel), len(delitos) - len(del_cel)
log.info("Join espacial: %d siniestros y %d delitos fuera de la cuadrícula", perdidos_v, perdidos_d)
print(f"Incidentes fuera del área de estudio -> siniestros: {perdidos_v} | delitos: {perdidos_d}")

## 5. Preparación de variables por celda (CRISP-DM · fase 3)

Se cuentan los incidentes por celda según las variables de la matriz de operacionalización.

> **Nota metodológica.** Una celda **sin registros** no significa **sin riesgo**: puede reflejar subregistro o falta de cobertura. Por eso se conserva todas las celdas (con 0) y se recomienda documentar la calidad de cada fuente (borrador de datos, sección *No olvidar*).

In [ ]:
conteos = pd.concat([
    sin_cel[sin_cel["lesionados"] > 0].groupby("id_celda").size().rename("siniestros_lesionados"),
    sin_cel[sin_cel["fallecidos"] > 0].groupby("id_celda").size().rename("siniestros_fallecidos"),
    del_cel[del_cel["tipo"] == "homicidio"].groupby("id_celda").size().rename("homicidios"),
    del_cel[del_cel["tipo"].isin(["robo_persona", "robo_domicilio"])].groupby("id_celda").size().rename("robos"),
], axis=1).reindex(grid["id_celda"]).fillna(0).astype(int)

tabla = (grid[["id_celda"]]
         .merge(conteos.reset_index(), on="id_celda", how="left")
         .merge(contexto, on="id_celda", how="left"))

# Control de calidad: valores faltantes en variables del índice
todas_vars = [v for lst in DIMENSIONES.values() for v in lst]
faltantes = tabla[todas_vars].isna().sum()
print("Valores faltantes por variable:"); print(faltantes[faltantes > 0] if faltantes.any() else "Ninguno")
tabla[todas_vars] = tabla[todas_vars].fillna(0)

tabla[todas_vars].describe().T.round(2)

## 6. Normalización (RF-05)

Las variables tienen unidades muy distintas (conteos, porcentajes, habitantes/km²). Se llevan a una escala común:

- **Min-max** (método principal): `x' = (x − mín) / (máx − mín)` → valores en [0, 1].
- **Z-score** (alternativa): `z = (x − media) / desviación` → útil para evaluar cuánto influye el método de normalización.

Para los conteos se aplica antes `log(1+x)` (parámetro `LOG_CONTEOS`), porque unas pocas celdas con muchos eventos aplastarían al resto.

In [ ]:
def normalizar(df, cols, metodo="minmax", log_conteos=LOG_CONTEOS):
    X = df[cols].astype(float).copy()
    if log_conteos:
        for c in cols:
            if c in VARS_CONTEO:
                X[c] = np.log1p(X[c])
    if metodo == "minmax":
        amp = (X.max() - X.min()).replace(0, np.nan)
        X = ((X - X.min()) / amp).fillna(0)
    elif metodo == "zscore":
        sd = X.std(ddof=0).replace(0, np.nan)
        X = ((X - X.mean()) / sd).fillna(0)
    else:
        raise ValueError("metodo debe ser 'minmax' o 'zscore'")
    return X

# Vista rápida del efecto de la normalización en una variable
ej = "robos"
fig, ax = plt.subplots(1, 3, figsize=(12, 3))
ax[0].hist(tabla[ej], bins=30); ax[0].set_title(f"{ej} (original)")
ax[1].hist(normalizar(tabla, [ej], "minmax")[ej], bins=30); ax[1].set_title("min-max (con log)")
ax[2].hist(normalizar(tabla, [ej], "zscore")[ej], bins=30); ax[2].set_title("z-score (con log)")
plt.tight_layout(); plt.show()

## 7. Cálculo del índice de criticidad (RF-06)

Se calcula en dos niveles:

1. **Puntaje de cada dimensión** = media de sus variables normalizadas.
2. **Índice de criticidad** = media ponderada de las cuatro dimensiones:

$$IC_i = \sum_{d} w_d \cdot D_{d,i}, \qquad \sum_d w_d = 1$$

donde $D_{d,i}$ es el puntaje de la dimensión $d$ en la celda $i$.

In [ ]:
def calcular_indice(df, pesos, metodo="minmax"):
    out = pd.DataFrame({"id_celda": df["id_celda"].values})
    for dim, cols in DIMENSIONES.items():
        out[dim] = normalizar(df, cols, metodo).mean(axis=1).values
    w = pd.Series(pesos, dtype=float)
    w = w / w.sum()                                   # los pesos siempre suman 1
    out["indice_criticidad"] = sum(out[d] * w[d] for d in DIMENSIONES)
    return out

res = calcular_indice(tabla, PESOS_DIM, "minmax")
assert (res["id_celda"].values == grid["id_celda"].values).all()   # alineado con la cuadrícula
log.info("Índice calculado (min-max). Media=%.3f, máx=%.3f", res["indice_criticidad"].mean(), res["indice_criticidad"].max())
res.describe().round(3)

## 8. Ranking y categorías por percentil (RF-07)

| Categoría | Criterio |
|---|---|
| **Crítico** | percentil ≥ 90 |
| **Alto** | percentil 75–90 |
| **Medio** | percentil 50–75 |
| **Bajo** | percentil < 50 |

In [ ]:
def clasificar(s):
    p = s.rank(pct=True, method="average")
    return pd.cut(p, [0, 0.50, 0.75, 0.90, 1.0], labels=["Bajo", "Medio", "Alto", "Crítico"], include_lowest=True)

res["ranking"] = res["indice_criticidad"].rank(ascending=False, method="first").astype(int)
res["categoria"] = clasificar(res["indice_criticidad"])

# Se anexan los datos originales para interpretar cada celda
ranking = res.merge(tabla, on="id_celda").sort_values("ranking")
print(res["categoria"].value_counts().reindex(["Crítico", "Alto", "Medio", "Bajo"]))
ranking.head(10)[["ranking", "id_celda", "indice_criticidad", "categoria"] + list(DIMENSIONES)].round(3)

In [ ]:
# Comparación de métodos de normalización: ¿cambia el orden de las zonas?
res_z = calcular_indice(tabla, PESOS_DIM, "zscore")
rho_norm, _ = spearmanr(res["indice_criticidad"], res_z["indice_criticidad"])
print(f"Spearman entre índice min-max e índice z-score: {rho_norm:.3f}")
log.info("Spearman min-max vs z-score: %.3f", rho_norm)

## 9. Análisis de sensibilidad (RF-08 · hipótesis H3)

Se modifica el peso de **cada dimensión** en ±10 %, ±20 % y ±30 % (y se renormalizan los pesos para que sumen 1). Luego se compara el nuevo ranking con el original con el **coeficiente de Spearman**:

- ρ cercano a 1 → el orden de las zonas casi no cambia (índice robusto).
- Además se reporta qué porcentaje de las celdas del **10 % más crítico** se mantiene.

In [ ]:
def sensibilidad(df, pesos_base, deltas=DELTAS):
    base = calcular_indice(df, pesos_base)["indice_criticidad"]
    n_top = max(1, int(round(len(base) * 0.10)))
    top_base = set(base.nlargest(n_top).index)
    filas = []
    for dim in pesos_base:
        for d in deltas:
            for signo in (+1, -1):
                p = dict(pesos_base); p[dim] *= (1 + signo * d)
                nuevo = calcular_indice(df, p)["indice_criticidad"]
                rho, _ = spearmanr(base, nuevo)
                top = set(nuevo.nlargest(n_top).index)
                filas.append({"dimension": dim, "delta": d, "signo": "+" if signo > 0 else "-",
                              "spearman": rho, "top10_conservado": len(top & top_base) / n_top})
    return pd.DataFrame(filas)

sens = sensibilidad(tabla, PESOS_DIM)
resumen = sens.groupby("delta").agg(spearman_min=("spearman", "min"), spearman_medio=("spearman", "mean"),
                                    top10_min=("top10_conservado", "min")).round(3)
resumen

In [ ]:
# Detalle: Spearman por dimensión y variación
pivot = sens.assign(escenario=lambda d: d["signo"] + (d["delta"] * 100).astype(int).astype(str) + "%") \
            .pivot(index="dimension", columns="escenario", values="spearman")
orden = [f"{s}{int(d*100)}%" for d in DELTAS for s in ("-", "+")]
pivot = pivot[orden]

fig, ax = plt.subplots(figsize=(8, 3))
im = ax.imshow(pivot.values, cmap="YlGn", vmin=min(0.8, pivot.values.min()), vmax=1)
ax.set_xticks(range(len(orden))); ax.set_xticklabels(orden)
ax.set_yticks(range(len(pivot))); ax.set_yticklabels(pivot.index)
for i in range(pivot.shape[0]):
    for j in range(pivot.shape[1]):
        ax.text(j, i, f"{pivot.values[i, j]:.3f}", ha="center", va="center", fontsize=8)
ax.set_title("Spearman vs ranking base"); plt.colorbar(im); plt.tight_layout(); plt.show()

rho_min_20 = sens.loc[np.isclose(sens["delta"], 0.20), "spearman"].min()
veredicto = "SE CUMPLE" if rho_min_20 >= UMBRAL_SPEARMAN else "NO se cumple"
print(f"H3 (piloto): Spearman mínimo con ±20 % = {rho_min_20:.3f} -> {veredicto} el umbral de {UMBRAL_SPEARMAN}")
log.info("H3: spearman_min(±20%%)=%.3f umbral=%.2f", rho_min_20, UMBRAL_SPEARMAN)

## 10. Análisis espacial: Moran global, Gi\* y LISA (sección 8.5 del anteproyecto)

- **Moran global:** resume si la criticidad tiende a agruparse (I > 0), dispersarse (I < 0) o ser aleatoria.
- **Getis-Ord Gi\*:** detecta *hotspots* y *coldspots* estadísticamente significativos.
- **Moran local (LISA):** clasifica cada celda como alto-alto (HH), bajo-bajo (LL) o atípica (HL, LH).

Se usan vecinos tipo **Queen** (celdas que comparten borde o vértice) con pesos estandarizados por fila.

In [ ]:
w = Queen.from_dataframe(grid, use_index=False)
w.transform = "r"
y = res["indice_criticidad"].values

mi = Moran(y, w, permutations=PERMUTACIONES)
print(f"Moran global I = {mi.I:.3f} | p (simulado) = {mi.p_sim:.4f}")

gi = G_Local(y, w, transform="R", star=True, permutations=PERMUTACIONES, seed=SEED)
res["hotspot_gi"] = np.select(
    [(gi.p_sim < 0.05) & (gi.Zs > 0), (gi.p_sim < 0.05) & (gi.Zs < 0)],
    ["Hotspot", "Coldspot"], default="No significativo")

lisa = Moran_Local(y, w, permutations=PERMUTACIONES, seed=SEED)
etiquetas = {1: "HH", 2: "LH", 3: "LL", 4: "HL"}
res["lisa"] = [etiquetas[q] if p < 0.05 else "No significativo" for q, p in zip(lisa.q, lisa.p_sim)]

log.info("Moran I=%.3f (p=%.4f) | hotspots Gi*=%d", mi.I, mi.p_sim, (res["hotspot_gi"] == "Hotspot").sum())
print(res["hotspot_gi"].value_counts().to_string())
print(res["lisa"].value_counts().to_string())

## 11. Validación y evaluación de hipótesis (CRISP-DM · fase 5)

### 11.1 H1 — Coincidencia con puntos críticos conocidos

Se verifica qué proporción de los puntos críticos conocidos cae dentro del 10 % y del 25 % de celdas con mayor criticidad. Como referencia, **al azar** se esperaría 10 % y 25 %, respectivamente.

In [ ]:
pc = gpd.sjoin(a_geodf(puntos_criticos), grid[["id_celda", "geometry"]], how="inner", predicate="within")
ids_pc = pc["id_celda"].unique()

def tasa_acierto(ids, q):
    umbral = res["indice_criticidad"].quantile(1 - q)
    top = set(res.loc[res["indice_criticidad"] >= umbral, "id_celda"])
    return np.mean([i in top for i in ids])

h1 = pd.DataFrame({"zona_considerada": ["Top 10 %", "Top 25 %"],
                   "aciertos": [tasa_acierto(ids_pc, 0.10), tasa_acierto(ids_pc, 0.25)],
                   "esperado_al_azar": [0.10, 0.25]}).round(3)
print(f"Puntos críticos ubicados en la cuadrícula: {len(ids_pc)}")
h1

### 11.2 H2 — Superposición espacial entre siniestralidad vial y criminalidad

Se miden dos cosas: la correlación de Spearman entre ambas dimensiones y el **Moran bivariado** (¿las celdas con alta peligrosidad vial están rodeadas de celdas con alta peligrosidad criminal?).

In [ ]:
rho_vc, p_vc = spearmanr(res["peligrosidad_vial"], res["peligrosidad_criminal"])
mbv = Moran_BV(res["peligrosidad_vial"].values, res["peligrosidad_criminal"].values, w, permutations=PERMUTACIONES)

print(f"Spearman (vial vs criminal): {rho_vc:.3f} (p = {p_vc:.2g})")
print(f"Moran bivariado: I = {mbv.I:.3f} (p simulado = {mbv.p_sim:.4f})")
log.info("H2: spearman=%.3f | moran_bv=%.3f (p=%.4f)", rho_vc, mbv.I, mbv.p_sim)

## 12. Visualización (RF-09)

### 12.1 Mapa estático

In [ ]:
mapa = grid.merge(res, on="id_celda")

fig, axes = plt.subplots(1, 2, figsize=(13, 7))
mapa.plot(column="indice_criticidad", cmap="YlOrRd", legend=True, ax=axes[0],
          legend_kwds={"shrink": 0.6, "label": "Índice de criticidad"})
axes[0].set_title("Índice de criticidad (piloto, datos sintéticos)")

colores = {"Hotspot": "#d7301f", "Coldspot": "#2b8cbe", "No significativo": "#eeeeee"}
mapa.plot(color=mapa["hotspot_gi"].map(colores), ax=axes[1], edgecolor="none")
axes[1].set_title("Hotspots / coldspots (Gi*, p < 0.05)")
for a in axes: a.set_axis_off()
plt.tight_layout(); plt.savefig(OUT / "mapa_estatico.png", bbox_inches="tight"); plt.show()

### 12.2 Mapa interactivo (Folium)

Incluye la capa de criticidad por celda, un mapa de calor y los puntos críticos conocidos. Se puede activar/desactivar cada capa desde el control superior derecho.

In [ ]:
mapa_ll = mapa.to_crs(CRS_GEO)
mapa_ll["indice_r"] = mapa_ll["indice_criticidad"].round(3)
centro = [(BBOX[1] + BBOX[3]) / 2, (BBOX[0] + BBOX[2]) / 2]

m = folium.Map(location=centro, zoom_start=11, tiles="cartodbpositron")

folium.Choropleth(
    geo_data=mapa_ll[["id_celda", "geometry"]].to_json(), data=mapa_ll, columns=["id_celda", "indice_criticidad"],
    key_on="feature.properties.id_celda", fill_color="YlOrRd", fill_opacity=0.7, line_opacity=0.05,
    legend_name="Índice de criticidad", name="Criticidad por celda",
).add_to(m)

folium.GeoJson(
    mapa_ll[["id_celda", "indice_r", "categoria", "ranking", "hotspot_gi", "geometry"]].astype({"categoria": str}).to_json(),
    name="Detalle (tooltip)", style_function=lambda f: {"fillOpacity": 0, "weight": 0},
    tooltip=folium.GeoJsonTooltip(fields=["id_celda", "ranking", "indice_r", "categoria", "hotspot_gi"],
                                  aliases=["Celda", "Ranking", "Índice", "Categoría", "Gi*"]),
).add_to(m)

cen = mapa_ll.geometry.centroid
HeatMap(list(zip(cen.y, cen.x, mapa_ll["indice_criticidad"])), radius=18, blur=22, name="Mapa de calor").add_to(m)

fg = folium.FeatureGroup(name="Puntos críticos conocidos (ficticios)")
for _, r in puntos_criticos.iterrows():
    folium.CircleMarker([r["lat"], r["lon"]], radius=5, color="#08519c", fill=True, tooltip=r["nombre"]).add_to(fg)
fg.add_to(m)

folium.LayerControl().add_to(m)
m.save(OUT / "mapa_interactivo.html")
m

## 13. Exportación y registro de ejecución (RF-10, RF-11)

In [ ]:
ranking.to_csv(OUT / "ranking_criticidad.csv", index=False, encoding="utf-8-sig")
sens.to_csv(OUT / "sensibilidad.csv", index=False)
mapa_ll.drop(columns=["indice_r"]).astype({"categoria": str}).to_file(OUT / "criticidad.geojson", driver="GeoJSON")

log.info("Exportados: ranking_criticidad.csv, sensibilidad.csv, criticidad.geojson, mapa_interactivo.html, mapa_estatico.png")
print("Archivos generados en", OUT.resolve())
for f in sorted(OUT.iterdir()): print(" -", f.name, f"({f.stat().st_size/1024:.0f} KB)")

# Subir el ranking a Supabase (activar GUARDAR_EN_SUPABASE = True en los parámetros)
if GUARDAR_EN_SUPABASE:
    cols_subir = ["id_celda", "ranking", "indice_criticidad", "categoria"] + list(DIMENSIONES)
    guardar_tabla_supabase(ranking[cols_subir].astype({"categoria": str}), TABLA_RANKING, columna_unica="id_celda")

# Para descargar todo en Colab:
# import shutil; shutil.make_archive("salidas_piloto", "zip", OUT)
# from google.colab import files; files.download("salidas_piloto.zip")

## 14. Extensión opcional: modelo predictivo base + informe con LLM

El **borrador de datos** propone una arquitectura híbrida: modelos de aprendizaje automático hacen la predicción y el LLM solo **explica** resultados. Esta sección es un esqueleto mínimo de esa idea; **no forma parte del alcance del índice** del anteproyecto, y sirve para probar el flujo.

> **Regla de oro:** el LLM no inventa probabilidades ni sustituye al modelo estadístico. Solo recibe cifras ya calculadas y las redacta.

### 14.1 Modelo predictivo base (Random Forest)

Se predice el número de delitos del **mes siguiente** en cada celda, usando rezagos mensuales y variables de contexto, con **división temporal** (se entrena con el pasado y se evalúa con el futuro).

⚠️ Los datos sintéticos no tienen dinámica temporal (las fechas son aleatorias), así que aquí el modelo aprenderá casi solo el patrón espacial. Con datos reales, la parte temporal será informativa.

In [ ]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error

# Panel celda x mes (se conservan los meses sin eventos como 0)
delitos_p = del_cel.copy()
delitos_p["mes"] = delitos_p["fecha"].dt.to_period("M").dt.to_timestamp()
meses = pd.date_range(delitos_p["mes"].min(), delitos_p["mes"].max(), freq="MS")
idx = pd.MultiIndex.from_product([grid["id_celda"], meses], names=["id_celda", "mes"])
panel = (delitos_p.groupby(["id_celda", "mes"]).size().rename("n_delitos")
         .reindex(idx, fill_value=0).reset_index().sort_values(["id_celda", "mes"]))

g = panel.groupby("id_celda")["n_delitos"]
panel["lag1"], panel["lag2"], panel["lag3"] = g.shift(1), g.shift(2), g.shift(3)
panel["media3"] = panel[["lag1", "lag2", "lag3"]].mean(axis=1)
panel["objetivo"] = g.shift(-1)                              # delitos del mes siguiente
panel["mes_sin"] = np.sin(2 * np.pi * panel["mes"].dt.month / 12)
panel["mes_cos"] = np.cos(2 * np.pi * panel["mes"].dt.month / 12)

estaticas = res[["id_celda", "exposicion", "vulnerabilidad", "peligrosidad_vial"]]
panel = panel.merge(estaticas, on="id_celda").dropna()

FEATURES = ["lag1", "lag2", "lag3", "media3", "mes_sin", "mes_cos", "exposicion", "vulnerabilidad", "peligrosidad_vial"]
CORTE = pd.Timestamp("2025-12-01")                           # entrenamiento < CORTE <= prueba
train, test = panel[panel["mes"] < CORTE], panel[panel["mes"] >= CORTE]
print(f"Entrenamiento: {len(train):,} filas | Prueba: {len(test):,} filas")

rf = RandomForestRegressor(n_estimators=200, max_depth=12, min_samples_leaf=5, n_jobs=-1, random_state=SEED)
rf.fit(train[FEATURES], train["objetivo"])
pred = rf.predict(test[FEATURES])

mae_rf = mean_absolute_error(test["objetivo"], pred)
mae_base = mean_absolute_error(test["objetivo"], test["media3"])
print(f"MAE Random Forest: {mae_rf:.3f} | MAE línea base (media 3 meses): {mae_base:.3f}")

# ¿El modelo detecta las celdas con más delitos? (10 % superior)
agg = test.assign(pred=pred).groupby("id_celda").agg(real=("objetivo", "sum"), pred=("pred", "sum"))
k = max(1, int(len(agg) * 0.10))
recall_k = len(set(agg["real"].nlargest(k).index) & set(agg["pred"].nlargest(k).index)) / k
print(f"Coincidencia del top 10 % predicho con el top 10 % real: {recall_k:.2%}")

pd.Series(rf.feature_importances_, index=FEATURES).sort_values().plot.barh(figsize=(6, 3), title="Importancia de variables (RF)")
plt.tight_layout(); plt.show()
# Para probar XGBoost: from xgboost import XGBRegressor y reemplazar el modelo (misma interfaz fit/predict).

### 14.2 Informe explicativo con LLM

Se construye un *prompt* con **hechos ya calculados** (ranking, dimensiones, hotspots, predicción) y se le pide al LLM que redacte un informe breve **sin añadir cifras**. Si no hay clave de API configurada, se genera un informe de plantilla con las mismas cifras, de modo que la celda funciona igualmente.

Para usar el LLM real: en Colab, panel izquierdo → 🔑 *Secretos* → crear `ANTHROPIC_API_KEY` y darle acceso al notebook.

In [ ]:
MODELO_LLM = "claude-sonnet-5-5"   # ajustar según el modelo disponible en la cuenta

pred_celda = agg["pred"].rename("pred_delitos_prox_meses")
top3 = ranking.head(3).merge(pred_celda, left_on="id_celda", right_index=True, how="left")
top3 = top3.merge(res[["id_celda", "hotspot_gi", "lisa"]], on="id_celda")

def hechos_celda(r):
    return {"celda": int(r.id_celda), "ranking": int(r.ranking), "indice_criticidad": round(float(r.indice_criticidad), 3),
            "categoria": str(r.categoria),
            "dimensiones": {d: round(float(r[d]), 3) for d in DIMENSIONES},
            "hotspot_gi": r.hotspot_gi, "lisa": r.lisa,
            "delitos_previstos_periodo_prueba": None if pd.isna(r.pred_delitos_prox_meses) else round(float(r.pred_delitos_prox_meses), 1)}

hechos = [hechos_celda(r) for _, r in top3.iterrows()]

SISTEMA = ("Eres un analista de riesgo urbano que redacta para autoridades municipales. Usa EXCLUSIVAMENTE las cifras del JSON. "
           "No inventes probabilidades, causas ni cifras. Si un dato es nulo, dilo. Indica que los datos del piloto son sintéticos si así se señala. "
           "Redacta en español, máximo 180 palabras, con una recomendación de verificación en campo por celda.")
PROMPT = ("Datos del piloto (SINTÉTICOS, no reales). Redacta un informe breve sobre las 3 celdas más críticas:\n"
          + json.dumps(hechos, ensure_ascii=False, indent=2))

def informe_plantilla(hechos):
    lineas = ["INFORME (plantilla sin LLM) — datos sintéticos de prueba"]
    for h in hechos:
        dom = max(h["dimensiones"], key=h["dimensiones"].get)
        lineas.append(f"- Celda {h['celda']}: puesto {h['ranking']}, índice {h['indice_criticidad']} ({h['categoria']}). "
                      f"Dimensión dominante: {dom}. Gi*: {h['hotspot_gi']}. Verificar en campo.")
    return "\n".join(lineas)

def informe_llm(prompt):
    try:
        from google.colab import userdata
        import anthropic
        cliente = anthropic.Anthropic(api_key=userdata.get("ANTHROPIC_API_KEY"))
        msg = cliente.messages.create(model=MODELO_LLM, max_tokens=600, system=SISTEMA,
                                      messages=[{"role": "user", "content": prompt}])
        return msg.content[0].text
    except Exception as e:
        log.warning("LLM no disponible (%s). Se usa plantilla.", type(e).__name__)
        return None

# Si se quiere el LLM real: !pip -q install anthropic  y configurar el secreto.
texto = informe_llm(PROMPT) or informe_plantilla(hechos)
print(texto)
(OUT / "informe_top3.txt").write_text(texto, encoding="utf-8")

## 15. Resumen y siguientes pasos

### Cumplimiento de requisitos funcionales en este piloto

| ID | Requisito | Estado |
|---|---|---|
| RF-01 a RF-03 | Carga de datos viales, criminales y contextuales | ✅ (sintéticos; plantilla para datos reales) |
| RF-04 | Georreferenciación con *join* espacial | ✅ |
| RF-05 | Normalización min-max y z-score | ✅ |
| RF-06 | Índice como media ponderada de dimensiones | ✅ |
| RF-07 | Ranking con percentiles | ✅ |
| RF-08 | Sensibilidad ±10/20/30 % con Spearman | ✅ |
| RF-09 | Mapas interactivos (Folium) y de calor | ✅ |
| RF-10 | Exportación CSV, GeoJSON y HTML | ✅ |
| RF-11 | Registro de ejecución | ✅ (`ejecucion.log`) |

### De piloto a proyecto real

1. **Datos reales:** solicitar/descargar siniestros (ATM), delitos (Policía Nacional/ECU911) y variables del INEC; verificar la cobertura 2023–2026 y reasignar los datos censales a la cuadrícula.
2. **Unidad de análisis:** probar también con los 67 circuitos policiales y comparar resultados con la cuadrícula.
3. **Pesos:** reemplazar los pesos provisionales por los definidos en el análisis multicriterio, y documentar la justificación.
4. **Etapas CRISP-DM del anteproyecto:** primero un subconjunto (2025), luego 2023–2026 completo y, por último, refinamiento.
5. **Empaquetado:** pasar las funciones de este notebook al módulo `risk_index.py` (alcance del proyecto) y versionarlo en GitHub.
6. **Calidad de datos:** documentar procedencia, unidad y subregistro por variable; recordar que ausencia de registros no es ausencia de riesgo.
7. **Fuera del alcance del anteproyecto:** dashboard completo, tiempo real con ECU911 y despliegue institucional (la sección 14 es solo una exploración opcional).